# Artifact Evaluation (Config-Driven)

Use one config cell to control exactly what gets evaluated and displayed.

This notebook is optimized for answering:
- Which runs/configurations performed best overall?
- Which runs performed best for a specific subject?
- What compact tables should be exported?

In [19]:
from pathlib import Path
import json
from typing import Any
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## 1. Evaluation Config

Edit only this cell when you want to run a different evaluation setup.

- Set which runs to compare
- Set the primary metric for ranking
- Toggle optional outputs
- Set a focus subject for subject-level analysis

In [20]:
RUN_IDS = [
  "20260408_1735_90a931e9",
  "20260408_1747_bce3c5ff",
  "20260408_1759_d0b4779b",
  "20260408_1811_f79c8bc1",
  "20260408_1824_eff5bd3a",
  "20260408_1835_c7a68041",
  "20260408_1849_89f91aac",
  "20260408_1908_cdfe1f4d",
  "20260408_1929_a944d1a5",
  "20260408_1943_ace09cf5",
  "20260408_1957_6fe9c54d",
  "20260408_2010_4c52a808",
  "20260408_2023_22bdb8dd",
  "20260408_2118_e55be510",
  "20260408_2205_3e41fbb1",
  "20260408_2238_d564163a",
  "20260408_2313_fd52d16c",
  "20260408_2338_da80acf9",
  "20260409_1106_e01ac839",
]

In [21]:
CONFIG = {
    # Paths
    "artifacts_root": str(Path.cwd().parent.parent / "artifacts"),
    "artifact_group": "lee-2019-fine-tuning",
    "artifact_paradigm": "SSVEP",

    # Runs to evaluate
    "run_ids": RUN_IDS,

    # Ranking and reporting
    "primary_metric": "mean_accuracy",  # mean_accuracy or mean_balanced_accuracy
    "top_k": 20,
    "show_config_columns": [
        "seed",
        "set_seed",
        "learning_rate",
        "batch_size",
        "set_eeg_reference",
        "convert_to_uV",
        "normalize",
        "normalization_method",
        "optimizer",
        "criterion",
        "requested_trial_duration_s",
        "subjects_to_use"
    ],

    # Subject-focused analysis
    "focus_subject": "51",  # str/int subject id, or None to disable
    "show_focus_subject_folds": True,

    # Optional outputs
    "show_run_summary_table": False,
    "show_subject_long_table": False,
    "show_fold_long_table": False,
    "export_tables": True,
}

In [22]:
ARTIFACTS_ROOT = Path(CONFIG["artifacts_root"])
RUN_IDS = CONFIG["run_ids"]
GROUP_DIR = ARTIFACTS_ROOT / CONFIG["artifact_group"] / CONFIG["artifact_paradigm"]

assert ARTIFACTS_ROOT.exists(), f"Artifacts root does not exist: {ARTIFACTS_ROOT}"
assert GROUP_DIR.exists(), f"Artifact group does not exist: {GROUP_DIR}"
assert RUN_IDS, "CONFIG['run_ids'] cannot be empty"

print("Artifacts root:", ARTIFACTS_ROOT)
print("Artifact group:", GROUP_DIR)
print("Run count:", len(RUN_IDS))
print("Primary metric:", CONFIG["primary_metric"])

Artifacts root: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts
Artifact group: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-fine-tuning/SSVEP
Run count: 19
Primary metric: mean_accuracy


## 2. Helpers

These helpers build compact tables for ranking and optional subject/fold deep dives.

In [23]:
def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def flatten_dict(d: dict[str, Any], parent_key: str = "", sep: str = ".") -> dict[str, Any]:
    out: dict[str, Any] = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, new_key, sep=sep))
        else:
            out[new_key] = v
    return out


def load_run_artifacts(group_dir: Path, run_id: str) -> dict[str, Any] | None:
    run_dir = group_dir / run_id
    if not run_dir.exists():
        print(f"  [SKIP] Run folder not found: {run_id}")
        return None

    files = {
        "config": run_dir / "config.json",
        "run_metadata": run_dir / "run_metadata.json",
        "global_metrics": run_dir / "global_metrics.json",
        "subject_metrics": run_dir / "subject_metrics.json",
        "cv_results": run_dir / "cv_results.json",
    }

    missing = [name for name, path in files.items() if not path.exists()]
    if missing:
        print(f"  [SKIP] Run {run_id} is missing artifact files: {missing}")
        return None

    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "config": read_json(files["config"]),
        "run_metadata": read_json(files["run_metadata"]),
        "global_metrics": read_json(files["global_metrics"]),
        "subject_metrics": read_json(files["subject_metrics"]),
        "cv_results": read_json(files["cv_results"]),
    }


def build_run_summary_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for run in runs:
        row = {
            "run_id": run["run_id"],
            "run_dir": str(run["run_dir"]),
        }
        row.update({f"config.{k}": v for k, v in flatten_dict(run["config"]).items()})
        row.update({f"meta.{k}": v for k, v in flatten_dict(run["run_metadata"]).items()})
        row.update({f"global.{k}": v for k, v in flatten_dict(run["global_metrics"]).items()})
        rows.append(row)
    return pd.DataFrame(rows)


def build_subject_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for run in runs:
        for subject_id, metrics in run["subject_metrics"].items():
            row = {
                "run_id": run["run_id"],
                "subject_id": str(subject_id),
            }
            row.update(flatten_dict(metrics))
            rows.append(row)
    return pd.DataFrame(rows)


def build_fold_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for run in runs:
        for fold in run["cv_results"]:
            row = {"run_id": run["run_id"]}
            row.update(fold)
            row["subject_id"] = str(row["subject_id"])
            rows.append(row)
    return pd.DataFrame(rows)


def order_by_run_ids(df: pd.DataFrame, run_ids: list[str]) -> pd.DataFrame:
    if "run_id" not in df.columns:
        return df
    out = df.copy()
    out["run_id"] = pd.Categorical(out["run_id"], categories=run_ids, ordered=True)
    out = out.sort_values("run_id").reset_index(drop=True)
    out["run_id"] = out["run_id"].astype(str)
    return out

## 3. Load and Build Core Tables

In [24]:
_raw = [load_run_artifacts(GROUP_DIR, run_id) for run_id in RUN_IDS]
runs = [r for r in _raw if r is not None]
_skipped = len(_raw) - len(runs)
if _skipped:
    print(f"Skipped {_skipped} run(s) (not yet on disk or incomplete).")

assert runs, "No runs were loaded. Check that run_ids exist under GROUP_DIR."

loaded_ids = [r["run_id"] for r in runs]
run_summary_df = order_by_run_ids(build_run_summary_df(runs), loaded_ids)
subject_df     = order_by_run_ids(build_subject_df(runs), loaded_ids)
fold_df        = order_by_run_ids(build_fold_df(runs), loaded_ids)

print(f"Loaded {len(runs)} / {len(RUN_IDS)} runs")
print(f"run_summary_df: {run_summary_df.shape}")
print(f"subject_df:     {subject_df.shape}")
print(f"fold_df:        {fold_df.shape}")

Loaded 19 / 19 runs
run_summary_df: (19, 41)
subject_df:     (133, 8)
fold_df:        (665, 14)


## 4. Overall Leaderboard (Best Runs/Configs)

This is the main output for comparing runs.

In [25]:
metric = CONFIG["primary_metric"]
metric_col = f"global.{metric}"
assert metric_col in run_summary_df.columns, f"Missing metric column: {metric_col}"

config_cols = [f"config.{k}" for k in CONFIG["show_config_columns"] if f"config.{k}" in run_summary_df.columns]

# Build leaderboard columns, removing duplicates while preserving order
primary_cols = [c for c in ["run_id", metric_col, "global.mean_accuracy", "global.mean_balanced_accuracy"] if c in run_summary_df.columns]
leaderboard_cols = list(dict.fromkeys(primary_cols + config_cols))

leaderboard_df = run_summary_df[leaderboard_cols].sort_values(metric_col, ascending=False).reset_index(drop=True)
top_k = min(CONFIG["top_k"], len(leaderboard_df))

print(f"Top {top_k} runs by {metric_col}:")
display(leaderboard_df.head(top_k))

if CONFIG["show_run_summary_table"]:
    display(run_summary_df)

Top 19 runs by global.mean_accuracy:


,run_id,global.mean_accuracy,global.mean_balanced_accuracy,config.seed,config.set_seed,config.learning_rate,config.batch_size,config.set_eeg_reference,config.convert_to_uV,config.normalize,config.normalization_method,config.optimizer,config.criterion,config.subjects_to_use
0,20260409_1106_e01ac839,0.922143,0.922143,5,False,NaN,32,True,True,False,None,Adam,None,None
1,20260408_1929_a944d1a5,0.920000,0.920000,5,True,0.0030,16,True,True,False,None,Adam,None,None
2,20260408_2010_4c52a808,0.918571,0.918571,5,True,0.0050,32,True,True,False,None,Adam,None,None
3,20260408_1943_ace09cf5,0.918571,0.918571,5,True,0.0030,32,True,True,False,None,Adam,None,None
4,20260408_1957_6fe9c54d,0.915000,0.915000,5,True,0.0050,16,True,True,False,None,Adam,None,None
5,20260408_1849_89f91aac,0.914286,0.914286,5,True,0.0010,16,True,True,False,None,Adam,None,None
6,20260408_2338_da80acf9,0.912857,0.912857,5,True,0.0005,32,True,True,False,None,Adam,None,None
7,20260408_2238_d564163a,0.912857,0.912857,5,True,0.0003,32,True,True,False,None,Adam,None,None
8,20260408_1908_cdfe1f4d,0.912143,0.912143,5,True,0.0010,32,True,True,False,None,Adam,None,None
9,20260408_2205_3e41fbb1,0.910714,0.910714,5,True,0.0003,16,True,True,False,None,Adam,None,None


## 5. Focus Subject Evaluation (Optional)

Set `focus_subject` in `CONFIG` to rank runs for one subject and inspect fold-level details.

In [26]:
focus_subject = CONFIG["focus_subject"]

if focus_subject is None:
    print("Focus subject analysis is disabled (focus_subject=None).")
else:
    focus_subject = str(focus_subject)
    subject_metric_col = "mean_accuracy"
    if CONFIG["primary_metric"] == "mean_balanced_accuracy" and "mean_balanced_accuracy" in subject_df.columns:
        subject_metric_col = "mean_balanced_accuracy"

    focus_subject_df = subject_df[subject_df["subject_id"] == focus_subject].copy()
    if focus_subject_df.empty:
        print(f"No subject metrics found for subject_id={focus_subject}")
    else:
        focus_subject_df = focus_subject_df.sort_values(subject_metric_col, ascending=False).reset_index(drop=True)
        top_k = min(CONFIG["top_k"], len(focus_subject_df))
        print(f"Top {top_k} runs for subject {focus_subject} by {subject_metric_col}:")
        display(focus_subject_df[[c for c in ["run_id", "subject_id", "mean_accuracy", "mean_balanced_accuracy", "std_accuracy"] if c in focus_subject_df.columns]].head(top_k))

        if CONFIG["show_focus_subject_folds"]:
            focus_folds = fold_df[fold_df["subject_id"] == focus_subject].copy()
            if focus_folds.empty:
                print(f"No fold-level rows found for subject_id={focus_subject}")
            else:
                fold_metric_col = "accuracy" if CONFIG["primary_metric"] == "mean_accuracy" else "balanced_accuracy"
                fold_metric_col = fold_metric_col if fold_metric_col in focus_folds.columns else "accuracy"
                focus_folds = focus_folds.sort_values(["run_id", fold_metric_col], ascending=[True, False]).reset_index(drop=True)
                print(f"Fold details for subject {focus_subject}:")
                display(focus_folds[[c for c in ["run_id", "subject_id", "fold_id", "accuracy", "balanced_accuracy", "best_valid_loss"] if c in focus_folds.columns]])

if CONFIG["show_subject_long_table"]:
    display(subject_df)

if CONFIG["show_fold_long_table"]:
    display(fold_df)

Top 19 runs for subject 51 by mean_accuracy:


,run_id,subject_id,mean_accuracy,mean_balanced_accuracy,std_accuracy
0,20260409_1106_e01ac839,51,0.655,0.655,0.076485
1,20260408_1929_a944d1a5,51,0.655,0.655,0.092736
2,20260408_1943_ace09cf5,51,0.650,0.650,0.044721
3,20260408_2010_4c52a808,51,0.645,0.645,0.062048
4,20260408_2238_d564163a,51,0.620,0.620,0.172772
5,20260408_1957_6fe9c54d,51,0.615,0.615,0.075166
6,20260408_2338_da80acf9,51,0.615,0.615,0.170000
7,20260408_1735_90a931e9,51,0.615,0.615,0.051478
8,20260408_1849_89f91aac,51,0.610,0.610,0.167780
9,20260408_1908_cdfe1f4d,51,0.605,0.605,0.166883


Fold details for subject 51:


,run_id,subject_id,fold_id,accuracy,balanced_accuracy,best_valid_loss
0,20260408_1735_90a931e9,51,3,0.675,0.675,0.590914
1,20260408_1735_90a931e9,51,2,0.650,0.650,0.906023
2,20260408_1735_90a931e9,51,1,0.625,0.625,0.968546
3,20260408_1735_90a931e9,51,5,0.600,0.600,1.109958
4,20260408_1735_90a931e9,51,4,0.525,0.525,0.804469
...,...,...,...,...,...,...
90,20260409_1106_e01ac839,51,5,0.750,0.750,0.897106
91,20260409_1106_e01ac839,51,4,0.700,0.700,0.849698
92,20260409_1106_e01ac839,51,1,0.675,0.675,1.001523
93,20260409_1106_e01ac839,51,3,0.625,0.625,0.911476


## 6. Export (Optional)

Exports compact tables so you can review top runs/configs outside the notebook.

In [27]:
if CONFIG["export_tables"]:
    export_dir = GROUP_DIR / "_comparison_exports"
    export_dir.mkdir(exist_ok=True)

    leaderboard_df.to_csv(export_dir / "leaderboard.csv", index=False)
    run_summary_df.to_csv(export_dir / "run_summary_full.csv", index=False)
    subject_df.to_csv(export_dir / "subject_summary_long.csv", index=False)
    fold_df.to_csv(export_dir / "fold_summary_long.csv", index=False)

    if CONFIG["focus_subject"] is not None:
        s = str(CONFIG["focus_subject"])
        focus_subject_df = subject_df[subject_df["subject_id"] == s]
        focus_folds_df = fold_df[fold_df["subject_id"] == s]
        focus_subject_df.to_csv(export_dir / f"subject_{s}_summary.csv", index=False)
        focus_folds_df.to_csv(export_dir / f"subject_{s}_folds.csv", index=False)

    print("Exported tables to:", export_dir)
else:
    print("Export is disabled in CONFIG.")

Exported tables to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-fine-tuning/SSVEP/_comparison_exports
